# Looking at LD

In [ ]:
library(dplyr)
library(ggplot2)
library(data.table)
library(stringr)

## get ld scores 


*Make one .ld file per chr per ancestry:*

*Make file "allsums_22.txt" for chr22 containing LD scores and variant counts per ancestry aggregated by MAF bin*

In [ ]:
# load in concatenated file for all variant LD scores
og <- fread('ldscores/afr_chr22.score.ld', stringsAsFactors = F)
names(og)<-c('variant', 'chr', 'bp', 'MAF', 'mean_rsq', 'variant_num', 'max_rsq', 'ldscore')
setorder(og, -ldscore)
afr_og <- og[, .SD[1], by = variant] # remove duplicate variants
afr_og <- data.frame(afr_og)

og <- fread('ldscores/amr_chr22.score.ld', stringsAsFactors = F)
names(og)<-c('variant', 'chr', 'bp', 'MAF', 'mean_rsq', 'variant_num', 'max_rsq', 'ldscore')
setorder(og, -ldscore)
amr_og <- og[, .SD[1], by = variant] # remove duplicate variants
amr_og <- data.frame(amr_og)

og <- fread('ldscores/eur_chr22.score.ld', stringsAsFactors = F)
names(og)<-c('variant', 'chr', 'bp', 'MAF', 'mean_rsq', 'variant_num', 'max_rsq', 'ldscore')
setorder(og, -ldscore)
eur_og <- og[, .SD[1], by = variant] # remove duplicate variants
eur_og <- data.frame(eur_og)


In [ ]:
limits <- c(0, 0.0004, 0.005,  0.02,  0.1, 0.3,  0.5)
afr_sums <- data.frame('MAF'=limits)
amr_all_sums <- data.frame('MAF'=limits)
eur_sums <- data.frame('MAF'=limits)

In [ ]:
for (i in 1:length(limits)) {
    afr_sums$ldscore[i] <- mean(filter(afr_og, afr_og$MAF>limits[i] & afr_og$MAF<limits[i+1])$ldsc)
    afr_sums$var_count[i] <- length(filter(afr_og, afr_og$MAF>limits[i] & afr_og$MAF<limits[i+1])$ldsc)
    
    amr_all_sums$ldscore[i] <- mean(filter(amr_og, amr_og$MAF>limits[i] & amr_og$MAF<limits[i+1])$ldsc)
    amr_sums$var_count[i] <- length(filter(amr_og, amr_og$MAF>limits[i] & amr_og$MAF<limits[i+1])$ldsc)
    
    eur_sums$ldscore[i] <- mean(filter(eur_og, eur_og$MAF>limits[i] & eur_og$MAF<limits[i+1])$ldsc)
    eur_sums$var_count[i] <- length(filter(eur_og, eur_og$MAF>limits[i] & eur_og$MAF<limits[i+1])$ldsc)
}

In [ ]:
allsums <- data.frame(afr_sums[,1:2], amr_sums[,2], eur_sums[,2], afr_sums[,3], amr_sums[,3], eur_sums[,3])
colnames(allsums) <- c('MAF', 'afr_aou_ld', 'amr_aou_ld', 'eur_aou_ld', 'afr_aou_count', 'amr_aou_count', 'eur_aou_count')
head(allsums)

In [76]:
write.table(allsums, 'ldscores/allsums_22.txt', sep='\t', quote=F, row.names=F)

In [ ]:
# relative LD score by variant count in MAF bin (since MAF bin sizes differ per ancestry)
afr <- allsums[,c(1:2,6)]
amr <- allsums[,c(1,3,7)]
eur <- allsums[,c(1,4,8)]

afr$afr_aou.rel_ld <- afr$afr_aou_ld/afr$afr_aou_count
amr$amrall_aou.rel_ld <- amr$amrall_aou_ld/amr$amrall_aou_count
eur$eur_aou.rel_ld <- eur$eur_aou_ld/eur$eur_aou_count

In [ ]:
write.table(afr, "afr_aou_22_ldscores.txt", row.names=F, quote=F)
write.table(amr, "amr_aou_22_ldscores.txt", row.names=F, quote=F)
write.table(eur, "eur_aou_22_ldscores.txt", row.names=F, quote=F)

## Plot LD (binned by MAF) coloured by functional annotation

In [81]:
limits <- c(0.0004, 0.005,  0.02,  0.1, 0.3,  0.5) # MAF limits
len_lim <- length(limits)
num_anc <- 3

In [ ]:
# plotting LD by ancestry, binned by MAF

# make DF
MAF <- c(rep(limits, num_anc))
ancestry <- c(rep("afr", len_lim), rep("amr", len_lim), rep("eur", len_lim))
ldsc <- c(afr$afr.rel_ld[1:len_lim], amr$amr.rel_ld[1:len_lim], eur$eur.rel_ld[1:len_lim])
varcount <- c(afr$afr_aou_count[1:len_lim], amr$amr_aou_count[1:len_lim],eur$eur_aou_count[1:len_lim])
data <- data.frame(MAF,ancestry,ldsc)

data$MAF_bin <- rep(1:len_lim, num_anc)
 
# plot
ldscounts_anc <- ggplot(data, aes(fill=ancestry, y=ldsc, x=MAF_bin)) + 
    geom_bar(position='dodge', stat="identity", width=0.9) +
    labs(
            title = "AoU LD scores by ancestry",
            x = "MAF bin upper limit",
            y = "Relative LD score"
        ) +
    scale_fill_manual(values = c("coral", "green3", "darkorchid"), name="Ancestry") +
    scale_x_discrete(limit=factor(limits)) + 
    theme_minimal(base_size=14)
ldscounts_anc

In [ ]:
ggsave(ldscounts_anc, 'ldscounts_anc.png', width=12, height=6)
write.table(data, "ld_bins_chr22.txt", quote=F, row.names=F)

## Plot LD (binned by MAF) coloured by functional annotation (in AFR)

In [ ]:
chr22_ld <- fread("ldscores/afr_chr22.score.ld", stringsAsFactors = F)
names(chr22_ld)<-c('variant', 'chr', 'bp', 'MAF', 'mean_rsq', 'variant_num', 'max_rsq', 'ldscore')
setorder(chr22_ld, -ldscore)
chr22_ld_filt <- chr22_ld[, .SD[1], by = variant] # remove duplicate variants

head(chr22_ld_filt)
dim(chr22_ld_filt)

SNP,chr,bp,MAF,mean_rsq,snp_num,max_rsq,ldscore
<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
22:16217089:T:G,22,16217089,0.492232,0.142038,3831,0.999086,545.146
22:16218751:G:A,22,16218751,0.492102,0.141933,3833,0.999020,545.027
22:16197423:A:T,22,16197423,0.492167,0.142116,3828,0.999505,545.020
22:16193140:A:C,22,16193140,0.492306,0.142036,3830,0.999371,544.999
22:16208941:A:G,22,16208941,0.492361,0.141923,3833,0.999086,544.992
22:16221337:T:C,22,16221337,0.492175,0.142028,3830,0.998938,544.968


[1] 1641726       8

In [ ]:
annots <- fread("annots/afr_chr22_annots.txt", stringsAsFactors = F, header=F)
names(annots)<-c('unmod')
head(annots)

In [ ]:
# extract variant name and annotation

annots$annot <- annots$unmod

annots$SNP <- str_extract(annots$unmod, "22:(.*)")  # variant name, everything after first :
annots$annot <- str_extract(annots$unmod,"^[^:]+") # annotation, everything after first :

annots <- annots[,2:3]
head(annots)

In [ ]:
annots$annot[grep("coding_l",annots$annot)] <- 'coding'
annots$annot[grep("coding_h",annots$annot)] <- 'coding'
annots$annot[grep("rna",annots$annot)] <- 'rna'
annots$annot[grep("proximal_utr",annots$annot)] <- 'proximal_utr'
annots$annot[grep("proximal_notutr",annots$annot)] <- 'proximal_notutr'
annots$annot[grep("intronic",annots$annot)] <- 'intronic'
annots$annot[grep("regulatory",annots$annot)] <- 'regulatory'
annots$annot[grep("other",annots$annot)] <- 'other'

head(annots)

In [ ]:
# add variant annotation category to LD score df
chr22_ld_annots <- merge(chr22_ld_filt, annots, by='variant')
head(chr22_ld_annots)

SNP,chr,bp,MAF,mean_rsq,snp_num,max_rsq,ldscore,annot
<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>
22:10534680:A:G,22,10534680,0.000130641,0.0154593,13,0.0330395,1.20097,other
22:10553700:C:G,22,10553700,0.000999234,0.0663602,20,0.2680490,2.32720,other
22:10553757:C:A,22,10553757,0.007716990,0.0736525,56,0.4009140,5.12454,other
22:10553774:A:G,22,10553774,0.006603970,0.0426937,35,0.2737800,2.49428,other
22:10553781:C:G,22,10553781,0.000927539,0.0143391,6,0.0203052,1.08603,other
22:10553788:G:T,22,10553788,0.002780070,0.0647742,20,0.3605370,2.29548,other


*Bin variants by MAF, calculate pooled relative LD score per functional annotation category*

In [ ]:
chr22_coding <- filter(chr22_ld_annots, chr22_ld_annots$annot=='coding')
chr22_rna <- filter(chr22_ld_annots, chr22_ld_annots$annot=='rna')
chr22_proximal_utr <- filter(chr22_ld_annots, chr22_ld_annots$annot=='proximal_utr')
chr22_proximal_notutr <- filter(chr22_ld_annots, chr22_ld_annots$annot=='proximal_notutr')
chr22_coding_intronic <- filter(chr22_ld_annots, chr22_ld_annots$annot=='coding_intronic')
chr22_regulatory <- filter(chr22_ld_annots, chr22_ld_annots$annot=='regulatory')
chr22_other <- filter(chr22_ld_annots, chr22_ld_annots$annot=='other')

In [ ]:
limits <- c(0,1e-04,2e-04,4e-04, 0.001, 0.005, 0.01, 0.02, 0.04, 0.1, 0.2, 0.3, 0.4, 0.5) # MAF limits
chr22_coding_sums <- data.frame('MAF'=limits)
chr22_rna_sums <- data.frame('MAF'=limits)
chr22_proximal_utr_sums <- data.frame('MAF'=limits)
chr22_proximal_notutr_sums <- data.frame('MAF'=limits)
chr22_coding_intronic_sums <- data.frame('MAF'=limits)
chr22_regulatory_sums <- data.frame('MAF'=limits)
chr22_other_sums <- data.frame('MAF'=limits)

dim(chr22_coding_sums)
length(limits)

In [ ]:
for (i in 1:13) {
    chr22_coding_sums$ldscore[i] <- mean(filter(chr22_coding, chr22_coding$MAF>limits[i] & chr22_coding$MAF<limits[i+1])$ldsc)
    chr22_coding_sums$var_count[i] <- length(filter(chr22_coding, chr22_coding$MAF>limits[i] & chr22_coding$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_rna_sums$ldscore[i] <- mean(filter(chr22_rna, chr22_rna$MAF>limits[i] & chr22_rna$MAF<limits[i+1])$ldsc)
    chr22_rna_sums$var_count[i] <- length(filter(chr22_rna, chr22_rna$MAF>limits[i] & chr22_rna$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_proximal_utr_sums$ldscore[i] <- mean(filter(chr22_proximal_utr, chr22_proximal_utr$MAF>limits[i] & chr22_proximal_utr$MAF<limits[i+1])$ldsc)
    chr22_proximal_utr_sums$var_count[i] <- length(filter(chr22_proximal_utr, chr22_proximal_utr$MAF>limits[i] & chr22_proximal_utr$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_proximal_notutr_sums$ldscore[i] <- mean(filter(chr22_proximal_notutr, chr22_proximal_notutr$MAF>limits[i] & chr22_proximal_notutr$MAF<limits[i+1])$ldsc)
    chr22_proximal_notutr_sums$var_count[i] <- length(filter(chr22_proximal_notutr, chr22_proximal_notutr$MAF>limits[i] & chr22_proximal_notutr$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_coding_intronic_sums$ldscore[i] <- mean(filter(chr22_coding_intronic, chr22_coding_intronic$MAF>limits[i] & chr22_coding_intronic$MAF<limits[i+1])$ldsc)
    chr22_coding_intronic_sums$var_count[i] <- length(filter(chr22_coding_intronic, chr22_coding_intronic$MAF>limits[i] & chr22_coding_intronic$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_regulatory_sums$ldscore[i] <- mean(filter(chr22_regulatory, chr22_regulatory$MAF>limits[i] & chr22_regulatory$MAF<limits[i+1])$ldsc)
    chr22_regulatory_sums$var_count[i] <- length(filter(chr22_regulatory, chr22_regulatory$MAF>limits[i] & chr22_regulatory$MAF<limits[i+1])$ldsc)
}
for (i in 1:13) {
    chr22_other_sums$ldscore[i] <- mean(filter(chr22_other, chr22_other$MAF>limits[i] & chr22_other$MAF<limits[i+1])$ldsc)
    chr22_other_sums$var_count[i] <- length(filter(chr22_other, chr22_other$MAF>limits[i] & chr22_other$MAF<limits[i+1])$ldsc)
}

head(chr22_coding_sums)

In [ ]:
# normalise LD scores by variant count
chr22_coding_sums$ldscore <- chr22_coding_sums$ldscore/chr22_coding_sums$var_count
chr22_rna_sums$ldscore <- chr22_rna_sums$ldscore/chr22_rna_sums$var_count
chr22_proximal_utr_sums$ldscore <- chr22_proximal_utr_sums$ldscore/chr22_proximal_utr_sums$var_count
chr22_proximal_notutr_sums$ldscore <- chr22_proximal_notutr_sums$ldscore/chr22_proximal_notutr_sums$var_count
chr22_coding_intronic_sums$ldscore <- chr22_coding_intronic_sums$ldscore/chr22_coding_intronic_sums$var_count
chr22_regulatory_sums$ldscore <- chr22_regulatory_sums$ldscore/chr22_regulatory_sums$var_count
chr22_other_sums$ldscore <- chr22_other_sums$ldscore/chr22_other_sums$var_count

In [ ]:
# dataframe for plotting
MAF <- c(rep(limits, 7))
annot <- c(rep("protein-coding", 14), rep("ncRNA", 14), rep("proximal UTR", 14), rep("proximal non-UTR", 14),
           rep("intronic", 14), rep("distal regulatory", 14), rep("un-annotated", 14))
ldsc <- c(chr22_coding_sums$ldscore, chr22_rna_sums$ldscore, chr22_proximal_utr_sums$ldscore, 
          chr22_proximal_notutr_sums$ldscore, chr22_coding_intronic_sums$ldscore,
          chr22_regulatory_sums$ldscore, chr22_other_sums$ldscore)
varcount <- c(chr22_coding_sums$var_count, chr22_rna_sums$var_count, chr22_proximal_utr_sums$var_count, 
          chr22_proximal_notutr_sums$var_count, chr22_coding_intronic_sums$var_count,
          chr22_regulatory_sums$var_count, chr22_other_sums$var_count)

data <- data.frame(MAF,annot,ldsc)
data$MAF_bin <- rep(1:14, 7)

In [ ]:
# plot
ldscounts_annot <- ggplot(data, aes(fill=annot, y=ldsc, x=MAF_bin)) + 
    geom_bar(position='dodge', stat="identity", width=0.9) + 
    scale_fill_manual(name="Functional annotation") +
    scale_x_discrete(limit=factor(limits)) + 
    labs(
            title = "AoU LD scores by functional annotation",
            x = "MAF bin upper limit",
            y = "Relative LD score"
        ) +
    theme_minimal(base_size=14)

ldscounts_annot

In [ ]:
ggsave(ldscounts_annot, 'ldscounts_annot.png', width=12, height=6)
write.table(data, "ld_bins_chr22_annotation.txt", quote=F, row.names=F)